# M08 — JSON anidado y schema que cambia (extra)

[← Anterior](../M02-ingesta-preparacion/04-lab-calidad-limpieza.ipynb) · [Siguiente →](02-lab-json-anidado-schema.ipynb)

**Extra.** El pipeline de pedidos (CSV → fact → Parquet) **no** pasa por aquí. Esto cubre lo que M02 no enseña: un JSON de **varios niveles**, bajarlo a columnas, enriquecerlo y **volver a un documento** que una app (o `mongoimport`) pueda comer. Y el otro dolor: el CRM de 2023 y el de 2024 **no tienen las mismas claves**.

En clase ejecutamos **este** fichero, de arriba abajo. El lab es donde construyes tú el mismo flujo sobre los dumps reales.

Ejecuta las celdas **aquí**, en este mismo fichero (clase, juntos). Va **montado**: explicación + código + lo que tienes que ver. Lo que construyes tú está en el **lab**.

Kernel: **Python (NovaShop)**.


## Arranque

La primera celda **no es Spark todavía**: busca la raíz del repo (aunque este notebook no esté en la carpeta de arriba) y deja `RAW`, `STAGING` y `CURATED` listos. La segunda pide una `SparkSession` en `local[*]` (todos los cores de esta máquina; no hay clúster).

Al ejecutar: rutas impresas y una versión `3.5.x` con master `local[*]`.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
# getOrCreate: si ya hay sesión en este kernel, la reusa (mismo puerto 4040)
spark = get_spark('novashop-clase-m08')
print(spark.version, spark.sparkContext.master)


## Qué hay en `data/raw/` (además de lo de siempre)

NovaShop “tuvo un CRM”:

| Fichero | Qué es | Filas |
|---------|--------|------:|
| `profiles_v1.jsonl` | Dump **2023**, plano (`fullName`, `country`, `email` string) | **100** (C0001–C0100) |
| `profiles_v2.jsonl` | Dump **2024**, anidado (`profile.contact.address.geo`…) | **200** (C0051–C0250) |

**50** clientes están en los dos (C0051–C0100): la migración se quedó a medias. v2 trae suciedad: 8 sin `address.country` (el país está en `profile.country`), 6 con `orders_preview` vacío, 5 sin email de trabajo.

`customers.csv` sigue siendo la ficha del pipeline. Estos JSON son **otra** fuente.


## Un documento v2 (para no perderse en el schema)

Esto es **un** cliente. Spark, al leer el JSONL, convierte cada llave anidada en `struct` y cada lista en `array`.

```json
{
  "customer_id": "C0051",
  "profile": {
    "contact": {
      "full_name": "Cliente 0051",
      "email": {"work": "c0051@novashop.test", "personal": null},
      "address": {
        "city": "Madrid",
        "country": "ES",
        "geo": {"lat": 40.42, "lon": -3.7}
      }
    },
    "prefs": {"channel": "web", "lang": "es"},
    "country": null
  },
  "orders_preview": [{"id": "X00510", "gmv": 12.5}],
  "meta": {"source": {"system": "crm", "version": 2}}
}
```

Niveles: documento → `profile` → `contact` → `address` → `geo`. Eso es lo que pedían “bastantes niveles”. `products.json` del M02 no llega ni a uno.


## Leer v2 e **inferir**: el schema *es* el árbol

JSONL: una línea = un objeto. **No** uses `multiLine` (eso era el array de productos).

Al ejecutar: `count` **200**. `printSchema()` enseña `struct` y `array`. Si ves todo `string` y ningún `struct`, no es este fichero.


In [ ]:
v2 = spark.read.json(str(RAW / "profiles_v2.jsonl"))
print("v2 filas", v2.count())
v2.printSchema()
v2.select("customer_id", "profile.contact.full_name", "profile.contact.address.geo.lat").show(3, truncate=False)


## Bajar un nivel: el punto (`a.b.c`) no explota filas

`col("profile.contact.address.country")` es **una columna**. Sigue habiendo 200 filas. El árbol no se copia a 200 × N productos.

Al ejecutar: 200 filas; algunos `country` nulos (los 8 sucios). `email.work` nulo en 5.


In [ ]:
from pyspark.sql.functions import col, coalesce, lit, size, explode, struct, to_json, when, row_number
from pyspark.sql.window import Window

print("filas", v2.count())
print(
    "address.country nulo",
    v2.where(col("profile.contact.address.country").isNull()).count(),
)  # 8
print(
    "email.work nulo",
    v2.where(col("profile.contact.email.work").isNull()).count(),
)  # 5
v2.select(
    "customer_id",
    col("profile.contact.address.country").alias("addr_country"),
    col("profile.country").alias("profile_country"),
).where(col("profile.contact.address.country").isNull()).show()


## `explode`: aquí **sí** cambian las filas

`orders_preview` es un array. `explode` convierte **cada elemento en una fila**. Un cliente con 3 previews pasa a 3 filas. Uno con lista vacía **desaparece** (`explode`); `explode_outer` lo deja con nulos.

Al ejecutar: más de 200 filas (casi todos tienen 1–3 previews; 6 tienen 0 y se caen con `explode`).


In [ ]:
prev = v2.select("customer_id", explode("orders_preview").alias("item"))
print("filas tras explode", prev.count())  # > 200
prev.select("customer_id", "item.id", "item.gmv").show(6, truncate=False)
print("clientes que se cayeron (preview vacío)", v2.count() - prev.select("customer_id").distinct().count())  # 6


## Contrato interno: una fila por cliente, columnas planas

Para *transformar* (nombres, país, recuentos) conviene **aplanar**. El país se rescata con el mismo truco que las fechas de M02: `coalesce` de dos sitios + `UNK`.

Al ejecutar: 200 filas; `country` nulo **0** (los 8 sucios salen del `profile.country`). `n_preview` 0 en 6 clientes.


In [ ]:
v2_flat = v2.select(
    col("customer_id"),
    col("profile.contact.full_name").alias("full_name"),
    coalesce(
        col("profile.contact.address.country"),
        col("profile.country"),
        lit("UNK"),
    ).alias("country"),
    col("profile.contact.email.work").alias("email_work"),
    col("profile.contact.address.city").alias("city"),
    size(col("orders_preview")).alias("n_preview"),
    lit("v2").alias("feed"),
)
print("country nulo", v2_flat.where(col("country").isNull()).count())  # 0
print("n_preview=0", v2_flat.where(col("n_preview") == 0).count())  # 6
v2_flat.show(5, truncate=False)


## Subir otra vez: `struct` + JSON (lo que “come” una app / Mongo)

La aplicación no quiere 8 columnas CSV. Quiere **el árbol**. Montas `struct(...)` y, si hace falta un string, `to_json`. Escribir JSONL es `write.json` (un objeto por fichero-partición; Spark deja un directorio).

Esto **no** es un connector Mongo. Es el documento enriquecido. Un `mongoimport` o un POST a un API usarían ese JSON.

Al ejecutar: una columna `doc` con llaves anidadas; el `show` recorta el string.


In [ ]:
nested = v2_flat.select(
    "customer_id",
    struct(
        struct(
            col("full_name"),
            struct(col("email_work").alias("work")).alias("email"),
            struct(col("city"), col("country")).alias("address"),
        ).alias("contact"),
        struct(col("n_preview").alias("preview_orders")).alias("stats"),
    ).alias("profile"),
    lit("novashop.profile.v2").alias("schema_id"),
)
nested.printSchema()
nested.select("customer_id", to_json(col("profile")).alias("profile_json")).show(2, truncate=80)

from paths import ensure_dirs

ensure_dirs()
dest = CURATED / "_demo_m08_profiles"
nested.write.mode("overwrite").json(str(dest))
print("escrito", dest)
print("releer", spark.read.json(str(dest)).count())  # 200


## Legacy: v1 no tiene `profile`

Mismo negocio, **otro contrato**. `fullName` vs `full_name`. `email` string vs `email.work`. Si haces `v1.union(v2)` a palo seco, Spark exige las mismas columnas en el mismo orden → peta o rellena basura.

Al ejecutar: v1 **100** filas, schema **plano** (todo al primer nivel).


In [ ]:
v1 = spark.read.json(str(RAW / "profiles_v1.jsonl"))
print("v1 filas", v1.count())
v1.printSchema()
v1.show(3, truncate=False)


## Unir las dos épocas: normalizas **cada** feed al mismo contrato

1. Aplanas v1 con los nombres **internos** (`full_name`, `country` vacío → nulo).
2. `unionByName(..., allowMissingColumns=True)` — las columnas que falten se crean nulas.
3. En el solape (50 ids) **gana v2** (`row_number` por `customer_id`, `feed` desc).

Al ejecutar: unión bruta 300 filas; después del “quédate con una ficha por id”: **250**. Eso son todos los clientes de NovaShop. Sin esto, o pierdes a C0001–C0050 (solo v1) o duplicas a C0051–C0100.


In [ ]:
v1_flat = v1.select(
    col("customer_id"),
    col("fullName").alias("full_name"),
    when(col("country") == "", None).otherwise(col("country")).alias("country"),
    col("email").alias("email_work"),
    lit(None).cast("string").alias("city"),
    lit(0).alias("n_preview"),
    lit("v1").alias("feed"),
)
bruto = v2_flat.unionByName(v1_flat)
print("unión bruta (con duplicados de solape)", bruto.count())  # 300
w = Window.partitionBy("customer_id").orderBy(col("feed").desc())  # v2 antes que v1
profiles = (
    bruto.withColumn("rn", row_number().over(w))
    .where(col("rn") == 1)
    .drop("rn")
)
print("una ficha por cliente", profiles.count())  # 250
print("vienen de v1", profiles.where(col("feed") == "v1").count())  # 50  (C0001–C0050)
print("vienen de v2", profiles.where(col("feed") == "v2").count())  # 200


Eso es “resolver el legacy cuando la migración no se hizo bien”: **un contrato interno**, `coalesce` de sitios distintos, y una regla de precedencia (aquí: el dump nuevo pisa al viejo). Spark no “adivina” el CRM; tú fijas qué gana.

**Siguiente:** [lab](02-lab-json-anidado-schema.ipynb) — creas el notebook y repites el flujo (con pruebas). El pipeline de pedidos no cambia.
